In [1]:
from sklearn.cluster import KMeans
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

In [2]:
df_base_churn = pd.read_csv('../data/pre-processed/data_telcom_customer_churn_ibm.csv')

In [3]:
df = df_base_churn.copy()


df['latitude'] = pd.to_numeric(
    df['latitude'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

df['longitude'] = pd.to_numeric(
    df['longitude'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

df['monthly_charges'] = pd.to_numeric(
    df['monthly_charges'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

df['total_charges'] = pd.to_numeric(
    df['total_charges'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

df['partner'] = df['partner'].map({'No': 0, 'Yes': 1})

df['dependents'] = df['dependents'].map({'No': 0, 'Yes': 1})

df['is_new_customer'] = (
    df['tenure_months'].isin([1, 2])
).astype(int)

df['phone_services'] = (
    df['phone_service'].map({'No': 0, 'Yes': 1})
)

df['multiples_lines'] = np.where(
    df['multiple_lines'].eq('Yes'),
    1,
    0
)

df['internet_dsl'] = (
    df['internet_service'] == 'DSL'
).astype(int)

df['internet_fiber'] = (
    df['internet_service'] == 'Fiber optic'
).astype(int)

df['internet_none'] = (
    df['internet_service'] == 'No'
).astype(int)

df['online_secutiry'] = (
    df['online_security'] == 'Yes'
).astype(int)

df['online_backup'] = (
    df['online_backup'] == 'Yes'
).astype(int)

df['device_protection'] = (
    df['device_protection'] == 'Yes'
).astype(int)

df['tech_support'] = (
    df['tech_support'] == 'Yes'
).astype(int)

df['streaming_tv'] = (
    df['streaming_tv'] == 'Yes'
).astype(int)

df['streaming_movies'] = (
    df['streaming_movies'] == 'Yes'
).astype(int)


df['contract_month_to_month'] = (
    df['contract'] == 'Month-to-month'
).astype(int)

df['contract_one_year'] = (
    df['contract'] == 'One year'
).astype(int)

df['contract_two_year'] = (
    df['contract'] == 'Two year'
).astype(int)


df['paperless_billing'] = (
    df['paperless_billing']
    .map({'No': 0, 'Yes': 1})
)


df['payment_method_mailed_check'] = (
    df['payment_method'] == 'Mailed check'
).astype(int)

df['payment_method_electronic_check'] = (
    df['payment_method'] == 'Electronic check'
).astype(int)

df['payment_method_bank_transfer_automatic'] = (
    df['payment_method'] == 'Bank transfer (automatic)'
).astype(int)

df['payment_method_credit_card_automatic'] = (
    df['payment_method'] == 'Credit card (automatic)'
).astype(int)


df['target'] = df['churn_value']


df['high_risk_profile'] = np.where(
    (df['is_new_customer'] == 1)
    & (df['contract_month_to_month'] == 1)
    & (df['internet_fiber'] == 1),
    1,
    0
)

df['avg_monthly_spend'] = np.where(
    df['tenure_months'] > 0,
    round(df['total_charges'] / df['tenure_months'], 2),
    df['monthly_charges']
)

df['num_services'] = (
    df['online_secutiry'].fillna(0)
    + df['online_backup'].fillna(0)
    + df['device_protection'].fillna(0)
    + df['tech_support'].fillna(0)
    + df['streaming_tv'].fillna(0)
    + df['streaming_movies'].fillna(0)
    + df['phone_services'].fillna(0)
    + df['multiples_lines'].fillna(0)
)

df['has_support'] = np.where(
    (df['online_secutiry'] == 1)
    | (df['tech_support'] == 1),
    1,
    0
)

df['num_supports'] = (
    df['online_secutiry'].fillna(0)
    + df['tech_support'].fillna(0)
)


df['new_customer_without_support'] = np.where(
    (df['tenure_months'].between(1, 5))
    & (df['num_supports'] == 0),
    1,
    0
)

df['tenure_support_ratio'] = (
    (df['tenure_months'] + 1)
    / (df['num_supports'] + 1)
)

df['fiber_without_support'] = np.where(
    (df['internet_fiber'] == 1)
    & (df['num_supports'] == 0),
    1,
    0
)

df['fiber_month_to_month'] = np.where(
    (df['internet_fiber'] == 1)
    & (df['contract_month_to_month'] == 1),
    1,
    0
)

In [4]:
df["total_charges"] = df["total_charges"].astype(str).str.strip()
df["total_charges"] = df["total_charges"].replace(["", "nan", "None"], np.nan)
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")

moda = df["total_charges"].mode()[0]

df["total_charges"] = df["total_charges"].fillna(moda)

df["avg_ticket"] = np.where(
    df["tenure_months"] > 0,
    df["total_charges"] / df["tenure_months"],
    0
)

df.head()

,customer_id,count,country,state,city,zip_code,lat_long,latitude,longitude,gender,...,high_risk_profile,avg_monthly_spend,num_services,has_support,num_supports,new_customer_without_support,tenure_support_ratio,fiber_without_support,fiber_month_to_month,avg_ticket
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,0,54.08,3,1,1,0,1.5,0,0,54.075000
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,1,75.82,1,0,0,1,3.0,1,1,75.825000
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,0,102.56,5,0,0,0,9.0,1,1,102.562500
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,0,108.79,6,1,1,0,14.5,0,1,108.787500
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,0,102.78,6,0,0,0,50.0,1,1,102.781633


In [5]:
geo_features = df[['latitude', 'longitude']].copy()

kmeans = KMeans(
    n_clusters=10,
    random_state=42,
    n_init=10
)

df['geo_cluster'] = kmeans.fit_predict(geo_features)

In [6]:
df.groupby('geo_cluster').agg(
    customers=('customer_id', 'count'),
    churn_rate=('target', 'mean')
).sort_values('churn_rate', ascending=False)

,customers,churn_rate
geo_cluster,,
9,256,0.289062
5,936,0.288462
4,688,0.273256
3,300,0.270000
6,1044,0.265326
8,316,0.262658
7,484,0.262397
1,2107,0.259136
0,404,0.247525


In [7]:
df.columns

Index(['customer_id', 'count', 'country', 'state', 'city', 'zip_code',
       'lat_long', 'latitude', 'longitude', 'gender', 'senior_citizen',
       'partner', 'dependents', 'tenure_months', 'phone_service',
       'multiple_lines', 'internet_service', 'online_security',
       'online_backup', 'device_protection', 'tech_support', 'streaming_tv',
       'streaming_movies', 'contract', 'paperless_billing', 'payment_method',
       'monthly_charges', 'total_charges', 'churn_label', 'churn_value',
       'churn_score', 'cltv', 'churn_reason', 'is_new_customer',
       'phone_services', 'multiples_lines', 'internet_dsl', 'internet_fiber',
       'internet_none', 'online_secutiry', 'contract_month_to_month',
       'contract_one_year', 'contract_two_year', 'payment_method_mailed_check',
       'payment_method_electronic_check',
       'payment_method_bank_transfer_automatic',
       'payment_method_credit_card_automatic', 'target', 'high_risk_profile',
       'avg_monthly_spend', 'num_serv

In [8]:
num_cols = [
    'monthly_charges',
    'total_charges',
    'avg_monthly_spend',
    'num_services',
    'num_supports',
    'tenure_support_ratio',
    'geo_cluster'
]

for col in num_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .replace('', None)
        .replace(' ', None)
        .str.replace(',', '.', regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

binary_cols = [
    'partner',
    'dependents',
    'is_new_customer',
    'phone_services',
    'multiples_lines',
    'internet_dsl',
    'internet_fiber',
    'internet_none',
    'online_secutiry',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'contract_month_to_month',
    'contract_one_year',
    'contract_two_year',
    'paperless_billing',
    'payment_method_mailed_check',
    'payment_method_electronic_check',
    'payment_method_bank_transfer_automatic',
    'payment_method_credit_card_automatic',
    'high_risk_profile',
    'has_support',
    'new_customer_without_support',
    'fiber_without_support',
    'fiber_month_to_month'
]

for col in binary_cols:
    df[col] = df[col].fillna(0).astype(int)

df['tenure_months'] = df['tenure_months'].astype(int)

features = [
    'partner',
    'dependents',
    'tenure_months',
    'is_new_customer',
    'phone_services',
    'multiples_lines',
    'internet_dsl',
    'internet_fiber',
    'internet_none',
    'online_secutiry',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'contract_month_to_month',
    'contract_one_year',
    'contract_two_year',
    'paperless_billing',
    'payment_method_mailed_check',
    'payment_method_electronic_check',
    'payment_method_bank_transfer_automatic',
    'payment_method_credit_card_automatic',
    'monthly_charges',
    'total_charges',
    'avg_monthly_spend',
    'num_services',
    'has_support',
    'num_supports',
    'high_risk_profile',
    'new_customer_without_support',
    'fiber_without_support',
    'fiber_month_to_month',
    'tenure_support_ratio',
    'geo_cluster'
]

X = df[features]
y = df['target']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
print(X.isna().sum())

partner                                   0
dependents                                0
tenure_months                             0
is_new_customer                           0
phone_services                            0
multiples_lines                           0
internet_dsl                              0
internet_fiber                            0
internet_none                             0
online_secutiry                           0
online_backup                             0
device_protection                         0
tech_support                              0
streaming_tv                              0
streaming_movies                          0
contract_month_to_month                   0
contract_one_year                         0
contract_two_year                         0
paperless_billing                         0
payment_method_mailed_check               0
payment_method_electronic_check           0
payment_method_bank_transfer_automatic    0
payment_method_credit_card_autom

In [11]:
import mlflow

print(mlflow.get_tracking_uri())

sqlite:///C:/Users/eduar/tech-challenge-fase1/notebooks/mlflow.db


In [12]:
models = {
    "LogReg": LogisticRegression(),
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier(), 
    "XGBClassifier": HistGradientBoostingClassifier()
}

for name, model in models.items():

    with mlflow.start_run(run_name=name):

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        accuracy = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_prob)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        mlflow.log_param("model_name", name)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model"
        )

        print(f"{name} salvo no MLflow!")

c:\Users\eduar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/06/09 22:13:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 22:13:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommen

PermissionError: [WinError 5] Acesso negado: '\\Users\\rodrigo'

In [ ]:
for name, model in models.items():
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f'\n{name}')
    print(classification_report(y_test, y_pred))
    print(f'ROC AUC: {roc_auc_score(y_test, y_prob)}')


LogReg
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.51      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

ROC AUC: 0.8413534320183937

DecisionTree
              precision    recall  f1-score   support

           0       0.83      0.80      0.82      1035
           1       0.50      0.55      0.52       374

    accuracy                           0.73      1409
   macro avg       0.66      0.67      0.67      1409
weighted avg       0.74      0.73      0.74      1409

ROC AUC: 0.6732103645147123

RandomForest
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.64      0.51      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70   